# marv-hyena on a TEXT Hyena model: StripedHyena-7B

Everything so far ran on **Evo 2**, a DNA model. This notebook runs the same tools on
**StripedHyena-Hessian-7B** (Together AI, 2023, Apache-2.0), an **English text** model built from the same
kind of blocks: 16 Hyena blocks alternating with 16 attention blocks. It is the architecture Evo 2 descends
from, trained on text instead of DNA.

| step | question | it works if |
|---|---|---|
| 1 | Does the model load and talk sense without flash-attn? | low loss on English, sensible next words |
| 2 | Do marv-hyena's tools agree with this model? | all 7 operator checks pass |
| 3 | How far do its Hyena filters reach, from the weights alone? | a reach table, in tokens |
| 4 | What do its gates look like on real text? | gate capture reconstructs every Hyena block |
| 5 | Repeat a list of words after a gap: which blocks let it copy? Do Hyena's **far** lags matter? | edits run, model stays healthy, hooks come off |
| 6 | Does input-dependent gating matter for predicting text? | gate freezes run |
| 7 | Every number in one results table | one row per measurement |

**How it runs without flash-attn:** the model's code imports flash-attn's attention module. marv-hyena puts a
stand-in with the same weights and maths in its place (PyTorch's built-in attention kernel), so nothing has
to compile. Step 1 checks that the model still predicts English well under it.

**Steps 5 and 6 are exploratory**: what they show is a candidate for a pre-registered prediction, not a
finding. **Runtime:** A100 + High-RAM; the first load downloads ~15 GB. Leave `QUICK = True` first.

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# ---- run settings: edit these lines only ----
QUICK = True          # True = small samples, to check everything runs. False = full sample sizes.
USE_DRIVE = False     # True = cache the 15 GB download and outputs on Google Drive.
SANITY_MAX_NLL = 4.0  # stop if the model's loss on plain English is worse than this (nats per token)

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
    os.makedirs(os.environ['HF_HOME'], exist_ok=True)
    OUT_ROOT = '/content/drive/MyDrive/marv_hyena'
else:
    OUT_ROOT = '/content/marv_hyena_out'
OUT = f"{OUT_ROOT}/text_sh7b_{'quick' if QUICK else 'full'}"
os.makedirs(OUT, exist_ok=True)
print('mode    =', 'QUICK' if QUICK else 'FULL')
print('outputs =', OUT)

In [ ]:
# Install marv-hyena (Colab already has torch, transformers, safetensors). No Evo 2, no flash-attn.
BRANCH = 'main'
import sys
if 'marv_hyena' in sys.modules:  # an old copy is already imported; a fresh clone would not replace it
    raise RuntimeError('marv-hyena is already imported in this session, so new code would not load. '
                       'Use Runtime > Restart session, then Run all.')
!rm -rf marv-hyena && git clone -q -b {BRANCH} https://github.com/thebnbrkr/marv-hyena.git
!pip install -q -r marv-hyena/requirements.txt safetensors transformers

import sys, subprocess, os
sys.path.insert(0, '/content/marv-hyena')
print('commit:', subprocess.run(['git', '-C', 'marv-hyena', 'rev-parse', '--short', 'HEAD'],
                                capture_output=True, text=True).stdout.strip())
assert os.path.exists('marv-hyena/marv_hyena/stripedhyena.py'), 'needs marv-hyena with stripedhyena.py'

In [ ]:
# marv-hyena's tests on tiny CPU models, including the StripedHyena-7B copy. Should say "133 passed" (or more).
!cd marv-hyena && python -m pytest -q

In [ ]:
# Load StripedHyena-Hessian-7B from the Hugging Face Hub (~15 GB the first time), bf16, onto the GPU.
import time, torch
import marv_hyena as mh
t0 = time.time()
hm = mh.HyenaModel.load_stripedhyena()
tok = hm.tokenizer
print(f'loaded in {time.time() - t0:.0f}s | marv-hyena {mh.__version__}')
print(hm.describe())
if torch.cuda.is_available():
    print(f'GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# Plain English to work with: Pride and Prejudice (public domain, Project Gutenberg).
import urllib.request, re
BOOK_PATH = 'data/pride_and_prejudice.txt'
os.makedirs('data', exist_ok=True)
if not os.path.exists(BOOK_PATH):
    urllib.request.urlretrieve('https://www.gutenberg.org/cache/epub/1342/pg1342.txt', BOOK_PATH)
BOOK = open(BOOK_PATH, encoding='utf-8').read()
start = BOOK.find('It is a truth universally acknowledged')
BOOK = BOOK[start if start > 0 else 0:]
BOOK = BOOK[: BOOK.find('*** END OF') if '*** END OF' in BOOK else len(BOOK)]
book_ids = tok(BOOK, add_special_tokens=False)['input_ids']
BOS = tok('')['input_ids'][:1]
WORDS = sorted(set(re.findall(r'\b[a-z]{4,9}\b', BOOK.lower())))
print(f'{len(BOOK):,} characters, {len(book_ids):,} tokens, {len(WORDS):,} distinct words')

In [ ]:
import json, random
from contextlib import ExitStack
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from marv_hyena.checks import run_operator_checks

MODEL_NAME, DATASET = 'StripedHyena-Hessian-7B', 'Pride and Prejudice (Gutenberg #1342)'
TABLE = mh.ResultsTable(f'{OUT}/table.jsonl')
RESULTS = {'mode': 'quick' if QUICK else 'full', 'model': MODEL_NAME}
HYENA, ATTN = hm.blocks_of('li'), hm.blocks_of('attn')
print('Hyena blocks:', HYENA)
print('attention blocks:', ATTN)

def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

def save():
    with open(f'{OUT}/results_text.json', 'w') as fh:
        json.dump(RESULTS, fh, default=jsonable, indent=1)

def boot_mean(vals, n_boot=4000, seed=0):
    vals = np.asarray(vals, float)
    if len(vals) < 2:
        return float(vals.mean()), (float('nan'), float('nan'))
    r = np.random.default_rng(seed)
    return float(vals.mean()), tuple(float(x) for x in np.percentile(
        [r.choice(vals, len(vals)).mean() for _ in range(n_boot)], [2.5, 97.5]))

def as_ids(token_list):
    return torch.tensor([token_list], dtype=torch.long, device=hm.device)

def run_with(contexts, ids):
    with ExitStack() as stack:
        for c in contexts:
            stack.enter_context(c)
        return hm.logits(ids)

def mean_nll(logits, ids, start, end):
    return -mh.span_logprob(logits, ids, start, end) / (end - start)

LEAD, N_WORDS, SKIP = 400, 30, 3  # word-list copy probe: text before the list, list length, tokens skipped

def word_copy_probe(gap, seed):
    rng = random.Random(seed)
    insert = tok(' ' + ' '.join(rng.sample(WORDS, N_WORDS)), add_special_tokens=False)['input_ids']
    off = (2_000 + 9_000 * seed) % max(1, len(book_ids) - (LEAD + gap + 60))
    bg = book_ids[off: off + LEAD + gap + 50]
    seq = BOS + bg[:LEAD] + insert + bg[LEAD:LEAD + gap] + insert + bg[LEAD + gap:]
    a0 = len(BOS) + LEAD
    b0 = a0 + len(insert) + gap
    return as_ids(seq), (a0, a0 + len(insert)), (b0, b0 + len(insert))

## 1. Does the model work without flash-attn?

Two checks that the loaded model is the real, working StripedHyena-7B:

- its average loss on 1,000 tokens of the novel should be low for a 7B model (around 2–3 nats per token);
  above `SANITY_MAX_NLL` the notebook stops, because the attention stand-in or the loading went wrong.
- its most likely next words after simple prompts should make sense.

Then **does more context help?** Over two 8,192-token stretches of the novel, the loss per 1,024-token slice
should fall as the model sees more text and then flatten. If it *rises* with position, long-range attention
is not working (for example a positional-encoding mismatch), and every long-gap result below is suspect.

In [ ]:
passage = as_ids(BOS + book_ids[5000:6024])
nll = mean_nll(hm.logits(passage), passage, 1, passage.shape[1])
print(f'loss on 1,024 tokens of the novel: {nll:.3f} nats/token (perplexity {np.exp(nll):.1f})')
RESULTS['sanity_nll'] = nll; save()
assert nll < SANITY_MAX_NLL, 'the model does not predict English well: loading or attention is broken -- stop'

for prompt in ['The capital of France is', 'The opposite of hot is', 'Monday, Tuesday, Wednesday,']:
    top = hm.logits(hm.ids(prompt))[-1].topk(5).indices.tolist()
    print(f'{prompt!r:35} -> ' + ' | '.join(repr(tok.decode([i])) for i in top))

ids = hm.ids('The capital of France is')
for _ in range(12):  # greedy, no cache: re-runs the sequence each step
    ids = torch.cat([ids, hm.logits(ids)[-1].argmax().view(1, 1)], dim=1)
print('greedy continuation:', repr(tok.decode(ids[0].tolist())))

In [ ]:
LP_WIN, SLICE = 8192, 1024
pos_rows = []
for s0 in (30_000, 90_000):
    wids = as_ids(BOS + book_ids[s0:s0 + LP_WIN])
    lp = mh.token_logprobs(hm.logits(wids), wids).cpu()  # lp[j] scores token j+1
    for a in range(0, LP_WIN, SLICE):
        pos_rows.append({'window': s0, 'positions': f'{a}-{a + SLICE}', 'nll': float(-lp[a:a + SLICE].mean())})
pos_df = pd.DataFrame(pos_rows).groupby('positions', sort=False).nll.mean().reset_index()
print(pos_df.round(3).to_string(index=False))
RESULTS['loss_by_position'] = pos_df.to_dict('records'); save()
first, last = pos_df.nll.iloc[0], pos_df.nll.iloc[-1]
print(f'first slice {first:.3f} -> last slice {last:.3f} nats/token: ' +
      ('more context helps, as it should' if last < first else 'MORE CONTEXT DOES NOT HELP -- treat long-gap results with suspicion'))

## 1b. Why does more context hurt?

**Ruled out so far.** Run 3: the rotary *pairing* (interleaved was far worse). Run 4: rotary *precision* (stored
bf16 frequencies change nothing; fully-bf16 angles make it worse) and *far* attention (a 1,024-token window
does not help).

**What run 4 showed.** With finer buckets, the damage starts at about position **256**, not 1,024:
2.61 nats/token over positions 0-128, 2.77 at 128-256, 3.53 at 256-512, 4.06 at 512-1,024, then flat at ~4.5.
It levels off where the Hyena filters run out (reach99 is ~650 tokens).

**Suspect now: compute precision.** The checkpoint repo also holds a "standalone" copy of the weights
(`pytorch-model.bin`) that stores every Hyena-block weight in float32, and the standalone code builds Hyena
blocks in float32. The float32 values are exactly the bf16 ones, so the *weights* are the same; what differs
is the precision of the *arithmetic*. The Hub wrapper runs everything in bf16 except poles and residues.
A Hyena output sums hundreds of past positions. If the useful signal is a small part of a large sum, bf16
rounding of that sum can swamp it, and more so as more past positions are summed, up to the filter's reach.
That fits "fine for ~256 tokens, then worse until ~650".

| setting | what changes |
|---|---|
| Hub default | everything bf16 (what runs now) |
| all float32 | every weight and activation float32 (~30 GB) |
| Hyena blocks float32 | only the 16 Hyena blocks; the residual stream between blocks stays bf16 |
| attention blocks float32 | only the 16 attention blocks |
| Hyena lags 256+ removed | locator: every Hyena filter cut at 256 tokens back (bf16) |
| attention window 256 | locator: attention sees only the last 256 tokens plus the first (bf16) |

If a float32 setting makes more context help, precision is the cause, and the partial settings say where. The
two locators say which mixer the damage comes through if precision is not it. Every setting is undone and the
model is checked to be unchanged.

In [ ]:
from contextlib import contextmanager, nullcontext
from marv_hyena import stripedhyena

DIAG_WINDOWS = (30_000, 90_000)
CONTEXTS = (0, 1024, 3072, LP_WIN - SLICE)
BUCKETS = ((0, 128), (128, 256), (256, 512), (512, 1024), (1024, 2048), (2048, 4096), (4096, 8192))
N_DIAG_SEEDS = 2 if QUICK else 5
H_BLOCKS, A_BLOCKS = hm.blocks_of('li'), hm.blocks_of('attn')

def by_position():
    lps = []
    for s0 in DIAG_WINDOWS:
        wids = as_ids(BOS + book_ids[s0:s0 + LP_WIN])
        lps.append(mh.token_logprobs(hm.logits(wids), wids).float().cpu())
    return pd.Series({f'{a}-{b}': float(-np.mean([float(lp[a:b].mean()) for lp in lps])) for a, b in BUCKETS})

def truncation():
    out = {}
    for ctx in CONTEXTS:
        vals = []
        for s0 in DIAG_WINDOWS:
            end = s0 + LP_WIN
            ids = as_ids(BOS + book_ids[end - SLICE - ctx:end])
            vals.append(float(-mh.token_logprobs(hm.logits(ids), ids)[-SLICE:].float().mean()))
        out[f'{ctx} tokens of context'] = float(np.mean(vals))
    return pd.Series(out)

def copy_baseline():
    out = {}
    for gap in (2000, 4000):
        accs = []
        for s in range(N_DIAG_SEEDS):
            pids, first, second = word_copy_probe(gap, s)
            accs.append(mh.span_accuracy(hm.logits(pids), pids, second[0] + SKIP, second[1]))
        out[f'copy@{gap}'] = float(np.mean(accs))
    return pd.Series(out)

@contextmanager
def hyena_cut(lag):
    with ExitStack() as st:
        for b in H_BLOCKS:
            st.enter_context(mh.band_ablate(hm, b, (lag, None), mode='zero'))
        yield

@contextmanager
def attention_window(w):
    stripedhyena.set_attention_window(hm, w)
    try:
        yield
    finally:
        stripedhyena.set_attention_window(hm, None)

HUB = 'Hub default'
SETTINGS = {
    HUB: nullcontext,
    'all float32': lambda: stripedhyena.compute_precision(hm, torch.float32),
    'Hyena blocks float32': lambda: stripedhyena.compute_precision(hm, torch.float32, blocks=H_BLOCKS),
    'attention blocks float32': lambda: stripedhyena.compute_precision(hm, torch.float32, blocks=A_BLOCKS),
    'Hyena lags 256+ removed': lambda: hyena_cut(256),
    'attention window 256': lambda: attention_window(256),
}
ref_ids = as_ids(BOS + book_ids[1000:1512])
ref = hm.logits(ref_ids)
dtypes = {n: p.dtype for n, p in hm.model.named_parameters()}
diag = {}
for name, setting in SETTINGS.items():
    t0 = time.time()
    with setting():
        diag[name] = {'by_position': by_position(), 'truncation': truncation(), 'copy': copy_baseline()}
    torch.cuda.empty_cache()
    print(f'{name:26} done in {time.time() - t0:4.0f}s')
assert torch.equal(ref, hm.logits(ref_ids)), 'a setting was not undone'
assert dtypes == {n: p.dtype for n, p in hm.model.named_parameters()}, 'a weight kept the wrong dtype'
print('every setting undone (checked: identical logits and weight dtypes)\n')

pd.set_option('display.width', 200)
for key, title in (('by_position', 'loss by position (nats/token)'),
                   ('truncation', 'loss on the last 1,024 tokens, by how much text comes before them'),
                   ('copy', 'word-list copy accuracy')):
    table = pd.DataFrame({name: d[key] for name, d in diag.items()})
    print(title); print(table.round(3).to_string()); print()

RESULTS['diagnostic_compute_precision'] = {name: {k: v.to_dict() for k, v in d.items()} for name, d in diag.items()}; save()
for name, d in diag.items():
    for label, val in d['truncation'].items():
        TABLE.add(model=MODEL_NAME, component='whole model', intervention=f'setting: {name}',
                  task=f'loss on last 1024 tokens, {label}', dataset=DATASET, n=len(DIAG_WINDOWS), effect=val,
                  unit='nats/token')

def gain(name):  # how much 7,168 tokens of context lower the loss vs none; positive = context helps
    t = diag[name]['truncation']
    return float(t.iloc[0] - t.iloc[-1])

for name in SETTINGS:
    print(f'{name:26} context gain {gain(name):+.3f} nats/token')
print()
fp32_fixes = [n for n in ('all float32', 'Hyena blocks float32', 'attention blocks float32') if gain(n) > 0.05]
late = lambda n: float(diag[n]['by_position'].iloc[-1])  # loss at positions 4096-8192
if gain(HUB) > 0.05:
    print('VERDICT: with the Hub default, more context helps. No long-context problem in this run.')
elif fp32_fixes:
    print(f'VERDICT: COMPUTE PRECISION. More context helps under: {", ".join(fp32_fixes)}.\n'
          'bf16 arithmetic is what breaks long context in this release. Run every long-context result in the\n'
          'smallest float32 setting that fixes it.')
else:
    print('VERDICT: float32 arithmetic does not fix it, so precision is not the cause.')
    for n in ('Hyena lags 256+ removed', 'attention window 256'):
        print(f'  {n}: loss at 4096-8192 {late(n):.3f} vs {late(HUB):.3f} default'
              + ('  <- this mixer carries the damage' if late(n) < late(HUB) - 0.3 else ''))

## 2. Do the tools agree with this model?

The model-agnostic checks: the residual writes add up to the real final state; the trace rows add up to the
real logit difference; for a Hyena block, the lag bands and the gate capture each reconstruct the operator,
and zeroing one band removes exactly that band's write; hooks come off. **If any fails, stop.**

In [ ]:
check_ids = as_ids(BOS + book_ids[20_000:22_048])
ok = run_operator_checks(hm, check_ids)
RESULTS['operator_checks_ok'] = ok; save()
assert ok, 'operator checks failed on StripedHyena-7B: do not trust anything below'
before = hm.logits(check_ids)

## 3. How far do the Hyena filters reach? (weights only)

Each Hyena filter is a sum of decaying (and oscillating) exponentials. The reach table reads them straight
from the weights: **reach99** is the lag, in **tokens**, within which 99% of a channel's filter weight lies
(median over channels), and **max99** is the longest-reaching channel. For comparison, most of Evo 2's LI
channels reach only 4–7 DNA letters.

In [ ]:
reach = mh.reach_table(hm, max_len=32_768)
mh.show_reach(reach)
RESULTS['reach'] = [{'block': r.block, 'reach50': r.reach50, 'reach90': r.reach90, 'reach99': r.reach99,
                     'max99': r.max_reach99} for r in reach]; save()
for r in reach:
    TABLE.add(model=MODEL_NAME, component=(r.block, 'mixer'), intervention='none', task='filter reach99 (median channel)',
              dataset='weights only', n=len(r.per_channel99), effect=r.reach99, unit='tokens',
              notes=f'max channel {r.max_reach99:.0f} tokens')
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot([r.block for r in reach], [r.reach99 for r in reach], 'o-', label='median channel (reach99)')
ax.plot([r.block for r in reach], [r.max_reach99 for r in reach], 's--', label='longest channel')
ax.set_yscale('log'); ax.set_xlabel('block'); ax.set_ylabel('tokens'); ax.legend()
ax.set_title('How far StripedHyena-7B Hyena filters reach'); plt.tight_layout(); plt.show()

# the per-block numbers are very similar, so look at the spread over channels inside a few blocks
fig, ax = plt.subplots(figsize=(9, 3))
for r in reach[:: max(1, len(reach) // 4)]:
    ax.hist(np.log10(np.maximum(r.per_channel99, 1)), bins=60, histtype='step', label=f'block {r.block}')
ax.set_xlabel('log10 reach99 of each channel (tokens)'); ax.set_ylabel('channels'); ax.legend()
plt.tight_layout(); plt.show()

## 4. What do the gates look like on real text?

Output gate **x2** and input gate **x1** at a few positions of the novel, per Hyena block. **rel_error** must
be small (x2 × convolution = real output). **closed share** = how often a channel's gate is near zero.
A description, not an estimate.

In [ ]:
POS = list(range(256, check_ids.shape[1], 256))
caps = mh.capture_gates(hm, check_ids, POS)
gate_rows = [{'block': b, 'rel_error': g.rel_error, 'closed_x2': float(g.closed_fraction('x2').mean()),
              'closed_x1': float(g.closed_fraction('x1').mean()),
              'mean_abs_x2': float(g.x2.abs().mean())} for b, g in caps.items()]
gates_df = pd.DataFrame(gate_rows)
print(gates_df.round(4).to_string(index=False))
assert gates_df.rel_error.max() < 5e-2, 'gate capture does not reconstruct some block'
RESULTS['gates'] = gate_rows; save()
for r in gate_rows:
    TABLE.add(model=MODEL_NAME, component=(r['block'], 'mixer'), intervention='none', task='gate closed share (x2)',
              dataset=DATASET, n=len(POS), effect=r['closed_x2'], unit='share of channels x positions',
              notes=f"rel_error={r['rel_error']:.1e}")

## 5. Copying a list of words across a gap: which blocks carry it?

A list of 30 random words from the novel is inserted into the text, then repeated after a **gap** of ordinary
text. The second time, the model can only predict the words by copying them from the first list. Measure:
next-token accuracy on the repeated list (first 3 tokens skipped).

| condition | what is removed |
|---|---|
| baseline | nothing |
| Hyena mixers ablated | every Hyena block's whole write (mean) |
| Hyena far lags removed | only lags ≥ 1024 tokens of every Hyena filter (mean) |
| Hyena near lags removed | only lags 1–127 tokens (mean), for comparison |
| attention mixers ablated | every attention block's whole write (mean), for reference |

**Health:** the loss on ordinary text before the list should stay near baseline, otherwise a drop in copying
only means the model broke.

In [ ]:
N_SEEDS = 2 if QUICK else 10
GAPS = (500, 1000, 2000, 4000) if QUICK else (500, 1000, 2000, 4000, 8000, 16000)
FAR, NEAR = (1024, None), (1, 127)

mean_ids = as_ids(BOS + book_ids[60_000:64_096])  # mean writes come from a different part of the book
MEANS_H = mh.mean_writes(hm, mean_ids, mh.mixers_of(hm, 'li'))
MEANS_A = mh.mean_writes(hm, mean_ids, mh.mixers_of(hm, 'attn'))
COPY_CONDS = {
    'baseline': lambda: [],
    'Hyena mixers ablated': lambda: [mh.mean_ablate(hm, MEANS_H)],
    'Hyena far lags (1024+) removed': lambda: [mh.band_ablate(hm, b, FAR) for b in HYENA],
    'Hyena near lags (1-127) removed': lambda: [mh.band_ablate(hm, b, NEAR) for b in HYENA],
    'attention mixers ablated': lambda: [mh.mean_ablate(hm, MEANS_A)],
}

copy_rows = []
t0 = time.time()
for gap in GAPS:
    for s in range(N_SEEDS):
        pids, first, second = word_copy_probe(gap, s)
        for name, make in COPY_CONDS.items():
            logits = run_with(make(), pids)
            copy_rows.append({'gap': gap, 'seed': s, 'condition': name,
                              'second_acc': mh.span_accuracy(logits, pids, second[0] + SKIP, second[1]),
                              'first_acc': mh.span_accuracy(logits, pids, first[0] + SKIP, first[1]),
                              'health_nll': mean_nll(logits, pids, 50, first[0])})
print(f'{len(copy_rows)} runs in {time.time() - t0:.0f}s')
RESULTS['copy'] = copy_rows; save()
assert torch.equal(before, hm.logits(check_ids)), 'a hook was left behind'

In [ ]:
cp = pd.DataFrame(copy_rows)
base_h = cp[cp.condition == 'baseline'].set_index(['gap', 'seed']).health_nll
summary = []
for (gap, cond), sub in cp.groupby(['gap', 'condition'], sort=False):
    acc, ci = boot_mean(sub.second_acc.values)
    s_idx = sub.set_index(['gap', 'seed'])
    dh = float((s_idx.health_nll - base_h.loc[s_idx.index]).mean())
    summary.append({'gap': gap, 'condition': cond, 'n': len(sub), 'copy_acc': acc, 'ci_low': ci[0], 'ci_high': ci[1],
                    'first_list_acc': float(sub.first_acc.mean()), 'health_change_nll': dh})
    TABLE.add(model=MODEL_NAME, component='all hyena mixers' if 'Hyena' in cond or cond == 'baseline' else 'all attn mixers',
              intervention=cond, task=f'word-list copy@{gap} tokens', dataset=DATASET, n=len(sub), effect=acc, ci=ci,
              unit='repeated-list token accuracy', notes=f'health change {dh:+.3f} nats/token', seeds=N_SEEDS)
copy_summary = pd.DataFrame(summary)
print(copy_summary.round(3).to_string(index=False))
RESULTS['copy_summary'] = summary; save()

conds = list(COPY_CONDS)
fig, ax = plt.subplots(figsize=(10, 3.5))
w = 0.8 / len(GAPS)
for i, gap in enumerate(GAPS):
    sub = copy_summary[copy_summary.gap == gap].set_index('condition').loc[conds]
    lo, hi = sub.ci_low.fillna(sub.copy_acc), sub.ci_high.fillna(sub.copy_acc)
    ax.bar(np.arange(len(conds)) + i * w, sub.copy_acc, w, yerr=[sub.copy_acc - lo, hi - sub.copy_acc],
           label=f'gap {gap:,} tokens')
ax.set_xticks(np.arange(len(conds)) + w * (len(GAPS) - 1) / 2, conds, rotation=15)
ax.set_ylabel('repeated-list accuracy'); ax.set_ylim(0, 1.05); ax.legend(); plt.tight_layout(); plt.show()
print('first_list_acc is the no-copy baseline (random words are hard to guess);')
print('health_change_nll should be near 0 for the band conditions; a large positive value means the model broke')

## 6. Does input-dependent gating matter for predicting text?

`freeze_gate` replaces a gate with its average over positions in **every** Hyena block, so those blocks apply
a fixed filter that no longer depends on the words. The measure is how much the loss on ordinary text rises
(nats per token; positive = worse).

In [ ]:
N_WIN, WIN, SCORE_FROM = (1 if QUICK else 4), 4096, 512
rng = random.Random(0)
starts = sorted(rng.sample(range(0, len(book_ids) - WIN), N_WIN))
FREEZE_CONDS = {
    'baseline': lambda: [],
    'Hyena x2 (output gate) frozen': lambda: [mh.freeze_gate(hm, b, 'x2') for b in HYENA],
    'Hyena x1 (input gate) frozen': lambda: [mh.freeze_gate(hm, b, 'x1') for b in HYENA],
}
fr = []
for w_i, s0 in enumerate(starts):
    wids = as_ids(BOS + book_ids[s0:s0 + WIN])
    for name, make in FREEZE_CONDS.items():
        fr.append({'window': w_i, 'condition': name,
                   'nll': mean_nll(run_with(make(), wids), wids, SCORE_FROM, wids.shape[1])})
assert torch.equal(before, hm.logits(check_ids)), 'a hook was left behind'
fdf = pd.DataFrame(fr)
base = fdf[fdf.condition == 'baseline'].set_index('window').nll
freeze_summary = []
for cond, sub in fdf[fdf.condition != 'baseline'].groupby('condition', sort=False):
    d = (sub.set_index('window').nll - base).values
    m, ci = boot_mean(d)
    freeze_summary.append({'condition': cond, 'n_windows': len(d), 'loss_increase': m, 'ci_low': ci[0], 'ci_high': ci[1]})
    TABLE.add(model=MODEL_NAME, component='all hyena mixers', intervention=cond, task='loss on text, change from baseline',
              dataset=DATASET, n=len(d), effect=m, ci=ci, unit='nats/token')
print(f"baseline loss: {base.mean():.3f} nats/token")
print(pd.DataFrame(freeze_summary).round(4).to_string(index=False))
RESULTS['freeze'] = freeze_summary; save()

## 7. The results table

Every number above, one row each, with interval, sample size and the marv-hyena commit. The same format as the
Evo 2 notebook's table, so DNA and text results can be compared by filtering one file each.

In [ ]:
rows = TABLE.rows()
print(f'{len(rows)} rows in {TABLE.path}')
cols = ['component', 'intervention', 'task', 'n', 'effect', 'ci_low', 'ci_high', 'unit']
print(pd.DataFrame(rows)[cols].round(3).to_string(index=False))
save()
if not USE_DRIVE:
    try:
        from google.colab import files
        files.download(str(TABLE.path)); files.download(f'{OUT}/results_text.json')
    except ImportError:
        pass